# 00 从一个 prompt 开始：最小推理闭环

今天只解决一个问题：

> 用户输入一句话，Qwen 是怎么返回一句回答的？

先不要管 SFT、LoRA、DPO。最底层闭环只有这条：

```text
用户 prompt
  -> scripts/infer.py
  -> tokenizer 把文字变数字
  -> model 加载 Qwen
  -> model.generate 生成新 token
  -> tokenizer.decode 变回中文回答
```

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)
print("本 micro notebook 默认只读代码；真实模型推理开关默认 False。")

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=80):
    lines = read_text(rel).splitlines()
    print(f"--- {rel}:{start}-{min(end, len(lines))} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = [i for i, line in enumerate(lines, start=1) if keyword in line]
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        show_file(rel, max(1, hit-context), min(len(lines), hit+context))
        print()

## 1. 这个闭环的入口是什么？

入口不是 notebook，也不是训练脚本，而是命令行调用 `scripts/infer.py`。

例如：

```powershell
python scripts/infer.py --prompt "请解释 LoRA" --local_files_only
```

这句话的意思是：把 prompt 交给 infer.py，让它负责加载模型并生成回答。

In [ ]:
find_lines("scripts/infer.py", "def parse_args", context=10)
find_lines("scripts/infer.py", "--prompt", context=5)

## 2. 这个闭环的出口是什么？

出口就是 `print(generate(args))` 打印出来的一段文本回答。

In [ ]:
find_lines("scripts/infer.py", "def main", context=8)
find_lines("scripts/infer.py", "print(generate(args))", context=4)

## 3. infer.py 的最小骨架

```text
parse_args 读用户输入
  -> load_model 加载 tokenizer 和 Qwen
  -> build_model_inputs 把 prompt 变成模型输入
  -> generate 调 model.generate 得到回答
```

In [ ]:
for keyword in ["def load_model", "def build_model_inputs", "def generate", "def main"]:
    find_lines("scripts/infer.py", keyword, context=2)

## 4. 输入和输出用小学生语言讲

输入：`prompt = "请解释 LoRA"`

中间：`文字 -> token ids -> Qwen 计算 -> 新 token ids`

输出：`一段中文回答`

## 5. 安全运行：默认不跑模型

下面这格默认不跑。你想亲眼看到回答时，把 `RUN_INFERENCE = True`。它只推理，不训练，不写项目文件。

In [ ]:
prompt = "请用一句话解释机器学习里的 LoRA。"
cmd = [sys.executable, "scripts/infer.py", "--prompt", prompt, "--max_new_tokens", "80", "--temperature", "0", "--local_files_only"]
print("命令:")
print(" ".join(cmd))
RUN_INFERENCE = False
if RUN_INFERENCE:
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, encoding="utf-8", errors="replace")
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
else:
    print("默认不运行模型。先理解闭环，再决定是否改 True。")

## 6. 本节面试三句话

1. 项目的最小入口是 `scripts/infer.py --prompt ...`。
2. 推理闭环是 tokenizer 编码、Qwen 生成、tokenizer 解码。
3. 后面的 SFT、LoRA、DPO 本质上都是为了改变这个最小推理闭环里的模型回答。